# Table of Contents

- [Connect](#Connect)
- [League Info](#League-Info)
- [Scraping Functions](#Scraping-Functions)
- [Run Scraping Code](#Run-Scraping-Code)
- [Clean Raw Data](#Clean-Raw-Data)
- [Lineups](#Lineups)

In [463]:
import asyncio
import re
import pandas as pd
import requests
import random
import numpy as np
import os
from pathlib import Path
from playwright.async_api import async_playwright
from datetime import datetime, date

#### Connect

In [489]:
async def connect_fotmob_tab():
    """
    Launches Firefox (not enterprise-managed) to avoid bot detection.
    """
    p = await async_playwright().start()
    browser = await p.firefox.launch(headless=False)
    context = await browser.new_context()
    page = await context.new_page()
    await page.goto("https://www.fotmob.com", wait_until="domcontentloaded")
    print("Connected to:", page.url)
    return p, browser, context, page

# Run once:
p, browser, context, page = await connect_fotmob_tab()

Connected to: https://www.fotmob.com/


#### League Info

In [490]:
def get_league_json(league_id: int, country_code: str = "USA", out_file: str | Path | None = None) -> dict:
    url = f"https://www.fotmob.com/api/leagues?id={league_id}&ccode3={country_code}"
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": f"https://www.fotmob.com/leagues/{league_id}",
        "Accept": "application/json",
    }
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    data = r.json()
    if out_file:
        out_file = Path(out_file)
        out_file.write_text(json.dumps(data, indent=2))
    return data


def match_ids_from_league_json_obj(league_json: dict) -> pd.DataFrame:
    matches = league_json["fixtures"]["allMatches"]
    rows = []
    for m in matches:
        rows.append({
            "match_id": m["id"],
            "home_team": m["home"]["name"],
            "away_team": m["away"]["name"],
            "home_team_id": m["home"]["id"],
            "away_team_id": m["away"]["id"],
            "utc_time": m.get("status", {}).get("utcTime"),
            "pageUrl": m.get("pageUrl"),   # relative
        })
    return pd.DataFrame(rows)


def build_match_url_map(matches_df: pd.DataFrame) -> dict[int, str]:
    """
    Build {match_id: full_match_url} using pageUrl from league JSON.
    Strips any existing hash and appends "#{match_id}:tab=stats" to avoid double-hash.
    """
    m = {}
    for _, r in matches_df.iterrows():
        mid = int(r["match_id"])
        pageUrl = r.get("pageUrl")
        if isinstance(pageUrl, str) and pageUrl.startswith("/"):
            base = f"https://www.fotmob.com{pageUrl}".split("#")[0]
            m[mid] = f"{base}#{mid}:tab=stats"
    return m


#### Scraping Functions

In [491]:
def parse_player_id(href: str):
    m = re.search(r"[?&]player=(\d+)", href or "")
    return int(m.group(1)) if m else None


async def safe_goto(page, url: str, retries: int = 2, timeout_ms: int = 60000):
    last_err = None

    for attempt in range(retries):
        try:
            await page.goto(url, wait_until="commit", timeout=timeout_ms)
            await page.wait_for_timeout(1500)
            return
        except Exception as e:
            last_err = e
            print(f"  goto retry {attempt + 1}/{retries} failed for {url}: {e}")
            await page.wait_for_timeout(2000)

    raise last_err


async def ensure_stats_tab(page):
    if await page.locator("text=Shot map").count() > 0:
        return
    if await page.locator("text=Top stats").count() > 0:
        return

    stats_btn = page.locator("button[class*='MatchNavButton']", has_text="Stats").first
    await stats_btn.wait_for(state="visible", timeout=15000)

    for _ in range(3):
        try:
            await stats_btn.click(timeout=5000)
        except Exception:
            await stats_btn.click(force=True, timeout=5000)

        try:
            await page.wait_for_selector("text=Shot map", timeout=6000)
            return
        except Exception:
            try:
                await page.wait_for_selector("text=Top stats", timeout=6000)
                return
            except Exception:
                await page.wait_for_timeout(800)

    await page.screenshot(path="debug_not_on_stats.png", full_page=True)
    raise RuntimeError("Could not switch to Stats tab (Shot map / Top stats never appeared).")


async def scroll_to_player_stats(page, timeout_ms=45000):
    start = asyncio.get_event_loop().time()
    header = page.locator("text=Player stats").first

    while (asyncio.get_event_loop().time() - start) * 1000 < timeout_ms:
        if await page.locator("button[class*='FilterButton']").count() > 0:
            return True

        if await header.count() > 0:
            try:
                await header.scroll_into_view_if_needed()
                await page.wait_for_timeout(700)
            except Exception:
                pass

        await page.mouse.wheel(0, 1400)
        await page.wait_for_timeout(600)

    return False


async def scrape_current_table(page, category: str) -> pd.DataFrame:
    row_locator = page.locator("tr[class*='TableRowStyled']")
    await row_locator.first.wait_for(timeout=25000)

    headers = []
    header_cells = page.locator("thead tr th")
    if await header_cells.count() > 0:
        for i in range(await header_cells.count()):
            headers.append((await header_cells.nth(i).inner_text()).strip())

    rows = []
    n_rows = await row_locator.count()

    for r in range(n_rows):
        tr = row_locator.nth(r)

        name_loc = tr.locator("span[class*='PlayerNameCSS']").first
        name = (await name_loc.inner_text()).strip() if await name_loc.count() else None

        link_loc = tr.locator("a[href*='?player=']").first
        href = await link_loc.get_attribute("href") if await link_loc.count() else None
        player_id = parse_player_id(href)

        tds = tr.locator("td")
        values = [(await tds.nth(j).inner_text()).strip() for j in range(await tds.count())]
        stats = values[1:]

        row = {
            "category": category,
            "player_name": name,
            "player_id": player_id,
            "href": href,
        }

        for i, val in enumerate(stats):
            col = headers[i + 1] if (headers and i + 1 < len(headers)) else f"stat_{i+1}"
            row[col] = val

        rows.append(row)

    return pd.DataFrame(rows)


async def scrape_match_player_tables(page, match_url: str, match_id: int | None = None) -> pd.DataFrame:
    await safe_goto(page, match_url)

    print(f"\nMATCH: {match_id or ''}  {match_url}")

    await ensure_stats_tab(page)

    ok = await scroll_to_player_stats(page, timeout_ms=45000)
    if not ok:
        await page.screenshot(path=f"debug_no_player_stats_{match_id or 'match'}.png", full_page=True)
        raise RuntimeError("Player stats section never appeared (FilterButton not found).")

    buttons = page.locator("button[class*='FilterButton']")
    await buttons.first.wait_for(state="visible", timeout=25000)

    labels = [(await buttons.nth(i).inner_text()).strip() for i in range(await buttons.count())]

    tables = []
    for label in labels:
        btn = page.locator("button[class*='FilterButton']", has_text=label).first
        await btn.click()
        await page.wait_for_timeout(600)

        df = await scrape_current_table(page, label)
        if match_id is not None:
            df.insert(0, "match_id", match_id)

        tables.append(df)
        print("  Scraped:", label, df.shape)

    final = pd.concat(tables, ignore_index=True)

    if match_id is not None:
        out_dir = Path("raw_data/match_reports")
        out_dir.mkdir(parents=True, exist_ok=True)
        fp = out_dir / f"{match_id}.pkl"
        final.to_pickle(fp)

    return final


async def scrape_many_matches(page, match_url_map: dict[int, str], match_ids: list[int], sleep_s=0.8) -> pd.DataFrame:
    dfs = []
    total = len(match_ids)

    for i, mid in enumerate(match_ids):
        url = match_url_map.get(int(mid))
        if not url:
            print(f"[{i+1}/{total}] Skipping (no url): {mid}")
            continue

        print(f"[{i+1}/{total}] {mid}")

        try:
            df = await scrape_match_player_tables(page, url, match_id=int(mid))
            dfs.append(df)
        except Exception as e:
            print(f"[{i+1}/{total}] FAILED: {mid} {e}")

        await asyncio.sleep(sleep_s)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


#### Run Scraping Code

In [492]:
season = '2026'
url = 'https://www.fotmob.com/api/leagues?id=9134&season={}'.format(season)
# r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
# league_json =r.json()
# matches_df = match_ids_from_league_json_obj(r.json())
# matches_df['match_date'] = pd.to_datetime(matches_df.utc_time).dt.date
# matches_df.to_csv('data/matches/{}.csv'.format(season), index=False)

matches_df = pd.read_csv('data/matches/2026.csv')
matches_df['match_date'] = pd.to_datetime(matches_df.utc_time).dt.date

In [493]:
sd = date(year=2026, month=4, day=2)
ed = date(year=2026, month=4, day=6)
mask = (matches_df.match_date >= sd) & (matches_df.match_date <= ed)
matches_df = matches_df[mask]

In [494]:
# Write active match IDs to file so load_to_db.py only loads these matches
active_ids = matches_df.match_id.astype(int).tolist()
Path("data").mkdir(exist_ok=True)
Path("data/active_match_ids.txt").write_text("\n".join(str(i) for i in active_ids))
print(f"Wrote {len(active_ids)} match IDs to data/active_match_ids.txt")

Wrote 8 match IDs to data/active_match_ids.txt


In [495]:
#matches_df = match_ids_from_league_json_obj(league_json)
match_url_map = build_match_url_map(matches_df)

In [496]:
t1 = datetime.now()
arr = list(matches_df.match_id)  # example
random.seed(42)
# x = random.sample(arr, k=50)
#long_all = await scrape_many_matches(page, match_url_map, x, sleep_s=0.8)
t2 = datetime.now()
print(t2-t1)

0:00:00.000363


In [497]:
arr

[5161481, 5161480, 5161482, 5161483, 5161484, 5161485, 5161486, 5161487]

In [499]:
t1 = datetime.now()

long_all = await scrape_many_matches(page, match_url_map, arr, sleep_s=0.8)

t2 = datetime.now()
print(t2 - t1)

[1/8] 5161481

MATCH: 5161481  https://www.fotmob.com/matches/san-diego-wave-fc-vs-boston-legacy-fc/1qkodmrs7#5161481:tab=stats
[1/8] FAILED: 5161481 Locator.wait_for: Timeout 25000ms exceeded.
Call log:
  - waiting for locator("tr[class*='TableRowStyled']").first to be visible

[2/8] 5161480

MATCH: 5161480  https://www.fotmob.com/matches/orlando-pride-vs-angel-city-fc/r7bo5rno#5161480:tab=stats


CancelledError: 

#### Clean Raw Data

In [500]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# CONFIG
# -----------------------------
IN_DIR = Path("raw_data/match_reports")
OUT_DIR = Path("data/match_reports")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ID_COLS = ["match_id", "player_id", "player_name", "href"]
LONG_KEY_COLS = ["match_id", "player_id", "player_name", "href", "category"]

priority = {
    "All": 0,
    "Top stats": 1,
    "Attack": 2,
    "Passes": 3,
    "Defense": 4,
    "Duels": 5,
    "Goalkeeping": 6,
    "1st": 7,
    "2nd": 8,
}

# -----------------------------
# NAME HELPERS
# -----------------------------
def snake(s: str) -> str:
    s = str(s).strip()
    s = s.replace("+", " plus ")
    s = re.sub(r"[()/]", " ", s)
    s = re.sub(r"[%]", " pct ", s)
    s = re.sub(r"[^A-Za-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_").lower()
    return s

def col_prefix(col: str) -> str:
    parts = col.split("__")
    return parts[0] if len(parts) > 1 else ""

def normalize_col(col: str) -> str:
    if col in ID_COLS:
        return col

    parts = col.split("__")
    if len(parts) == 1:
        return snake(col)

    # Keep only the final metric token (drop category prefix)
    metric = parts[-1]
    return snake(metric)

# -----------------------------
# 1) LONG -> WIDE (one row per player)
# -----------------------------
def long_to_wide_per_player(df: pd.DataFrame) -> pd.DataFrame:
    if "category" not in df.columns:
        if "match_id" in df.columns and "player_id" in df.columns:
            return df.drop_duplicates(subset=["match_id", "player_id"], keep="first").copy()
        return df.copy()

    for c in ID_COLS:
        if c not in df.columns:
            df[c] = pd.NA

    join_keys = ["match_id", "player_id"]

    base = (
        df.sort_values(join_keys)
          .groupby(join_keys, as_index=False)
          .agg({"player_name": "first", "href": "first"})
    )

    cats = df["category"].dropna().unique().tolist()
    wide = base

    for cat in cats:
        sub = df[df["category"] == cat].copy()
        sub = sub.drop_duplicates(subset=join_keys, keep="first")

        value_cols = [c for c in sub.columns if c not in LONG_KEY_COLS]
        if not value_cols:
            continue

        sub = sub[join_keys + value_cols]
        sub = sub.rename(columns={c: f"{cat}__{c}" for c in value_cols})

        wide = wide.merge(sub, on=join_keys, how="left")

    ordered = ["match_id", "player_id", "player_name", "href"]
    rest = [c for c in wide.columns if c not in ordered]
    return wide[ordered + rest]

# -----------------------------
# 2) DEDUPE METRICS + FINAL RENAME
# -----------------------------
def dedupe_and_normalize_columns(df: pd.DataFrame):
    for c in ID_COLS:
        if c not in df.columns:
            df[c] = pd.NA

    norm_map = {c: normalize_col(c) for c in df.columns}

    groups = {}
    for orig, norm in norm_map.items():
        groups.setdefault(norm, []).append(orig)

    clean = df[ID_COLS].copy()
    report_rows = []

    for norm_name, cols in groups.items():
        if norm_name in ID_COLS:
            continue

        if len(cols) == 1:
            clean[norm_name] = df[cols[0]]
            continue

        def pr(c):
            return priority.get(col_prefix(c), 999), c

        cols_sorted = sorted(cols, key=pr)

        combined = df[cols_sorted[0]].copy()
        for c in cols_sorted[1:]:
            combined = combined.combine_first(df[c])

        conflict_count = 0
        primary = df[cols_sorted[0]]
        for c in cols_sorted[1:]:
            mask = primary.notna() & df[c].notna() & (primary.astype(str) != df[c].astype(str))
            conflict_count += int(mask.sum())

        clean[norm_name] = combined

        report_rows.append({
            "normalized_metric": norm_name,
            "original_columns": cols_sorted,
            "chosen_primary": cols_sorted[0],
            "conflicting_cells_vs_primary": conflict_count
        })

    if report_rows:
        report = pd.DataFrame(report_rows).sort_values(
            ["conflicting_cells_vs_primary", "normalized_metric"],
            ascending=[False, True]
        )
    else:
        report = pd.DataFrame(columns=[
            "normalized_metric",
            "original_columns",
            "chosen_primary",
            "conflicting_cells_vs_primary"
        ])

    if "match_id" in clean.columns and "player_id" in clean.columns:
        clean = clean.drop_duplicates(subset=["match_id", "player_id"], keep="first")

    return clean, report

# -----------------------------
# 3) FRACTION PARSER: "1/5 (20%)" -> *_succeeded, *_attempted, *_pct
# -----------------------------
_frac_pat = re.compile(r"^\s*(\d+)\s*/\s*(\d+)\s*\((\d+(?:\.\d+)?)%\)\s*$")

def split_fraction_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in list(out.columns):
        if col in ID_COLS:
            continue

        # only try on object-ish cols
        if out[col].dtype != object:
            continue

        ser = out[col].astype(str)
        mask = ser.str.match(_frac_pat) & ~ser.str.lower().eq("nan")

        # only split if at least one value matches
        if not mask.any():
            continue

        extracted = ser.where(mask).str.extract(_frac_pat)
        # extracted columns: 0=succeeded, 1=attempted, 2=pct

        out[f"{col}_succeeded"] = pd.to_numeric(extracted[0], errors="coerce")
        out[f"{col}_attempted"] = pd.to_numeric(extracted[1], errors="coerce")
        out[f"{col}_pct"] = pd.to_numeric(extracted[2], errors="coerce") / 100.0

        # drop original ONLY if every non-null matches the pattern
        non_null = out[col].notna()
        if (mask | ~non_null).all():
            out = out.drop(columns=[col])

    return out

# -----------------------------
# MAIN LOOP
# -----------------------------
match_ids_to_clean = set(matches_df.match_id.astype(int))
pkls = sorted(p for p in IN_DIR.glob("*.pkl") if int(p.stem) in match_ids_to_clean)
print(f"Cleaning {len(pkls)} files for current date range")

all_reports = []

for pkl_path in pkls:
    df = pd.read_pickle(pkl_path)

    match_id = None
    if "match_id" in df.columns and df["match_id"].notna().any():
        try:
            match_id = int(pd.to_numeric(df["match_id"], errors="coerce").dropna().iloc[0])
        except Exception:
            match_id = None
    if match_id is None:
        match_id = pkl_path.stem

    wide_df = long_to_wide_per_player(df)
    clean_df, report_df = dedupe_and_normalize_columns(wide_df)

    # NEW: split fraction columns into succeeded/attempted/pct
    clean_df = split_fraction_columns(clean_df)

    out_file = OUT_DIR / f"{match_id}.csv"
    clean_df.to_csv(out_file, index=False)

    if not report_df.empty:
        report_df.insert(0, "match_id", match_id)
        report_df.insert(1, "source_file", pkl_path.name)
        all_reports.append(report_df)

    print(f"Saved {out_file}  rows={clean_df.shape[0]} cols={clean_df.shape[1]}")

if all_reports:
    all_reports_df = pd.concat(all_reports, ignore_index=True)
    report_file = OUT_DIR / "duplicate_metric_report_all_matches.csv"
    all_reports_df.to_csv(report_file, index=False)
    print(f"\nSaved combined report: {report_file}")
else:
    print("\nNo duplicate metric conflicts found.")

Cleaning 5 files for current date range
Saved data/match_reports/5161483.csv  rows=32 cols=60
Saved data/match_reports/5161484.csv  rows=32 cols=60
Saved data/match_reports/5161485.csv  rows=30 cols=60
Saved data/match_reports/5161486.csv  rows=29 cols=60
Saved data/match_reports/5161487.csv  rows=22 cols=54

Saved combined report: data/match_reports/duplicate_metric_report_all_matches.csv


### Lineups

In [458]:
import asyncio
import pandas as pd
from pathlib import Path

LINEUP_DIR = Path("data/lineups")
LINEUP_DIR.mkdir(parents=True, exist_ok=True)


def _get_lineup_block(md: dict):
    if isinstance(md.get("lineup"), dict):
        return md["lineup"]
    if isinstance(md.get("content", {}).get("lineup"), dict):
        return md["content"]["lineup"]
    return None


def extract_lineup_rows(md: dict):
    lineup = _get_lineup_block(md)
    if not lineup:
        return []

    match_id = lineup.get("matchId") or md.get("general", {}).get("matchId")

    def pick(team, bucket):
        if bucket == "starters":
            return team.get("starters") or team.get("startingPlayers") or []
        if bucket == "bench":
            return team.get("bench") or team.get("subs") or team.get("substitutes") or []
        return []

    def team_rows(team, side):
        rows = []
        for bucket in ["starters", "bench"]:
            for p in pick(team, bucket):
                rows.append({
                    "match_id": match_id,
                    "side": side,
                    "bucket": bucket,
                    "team_id": team.get("id"),
                    "team_name": team.get("name"),
                    "formation": team.get("formation"),
                    "player_id": p.get("id"),
                    "player_name": p.get("name"),
                    "shirt_number": p.get("shirtNumber"),
                    "position_id": p.get("positionId"),
                    "usual_position_id": p.get("usualPlayingPositionId"),
                    "rating": (p.get("performance") or {}).get("rating"),
                    "h_x": (p.get("horizontalLayout") or {}).get("x"),
                    "h_y": (p.get("horizontalLayout") or {}).get("y"),
                    "v_x": (p.get("verticalLayout") or {}).get("x"),
                    "v_y": (p.get("verticalLayout") or {}).get("y"),
                })
        return rows

    home = lineup.get("homeTeam") or {}
    away = lineup.get("awayTeam") or {}

    return team_rows(home, "home") + team_rows(away, "away")


async def fetch_match_details_via_page_fetch(page, match_id: int) -> dict:
    """
    Uses the authenticated/verified browser session (cookies, Turnstile pass, etc.)
    to fetch FotMob JSON directly.
    """
    url = f"https://www.fotmob.com/api/data/matchDetails?matchId={match_id}"
    return await page.evaluate(
        """async (url) => {
            const res = await fetch(url, { credentials: "include" });
            if (!res.ok) {
                const txt = await res.text();
                return { _error: true, status: res.status, body: txt.slice(0, 500) };
            }
            return await res.json();
        }""",
        url,
    )


async def scrape_lineups_fast(page, match_ids, sleep_s=0.15, overwrite=False):
    """
    Saves one pickle per match_id in raw_data/lineups/<match_id>.pkl
    """
    for mid in match_ids:
        out = LINEUP_DIR / f"{int(mid)}.pkl"
        if out.exists() and not overwrite:
            print("Skip (exists):", mid)
            continue

        md = await fetch_match_details_via_page_fetch(page, int(mid))

        if isinstance(md, dict) and md.get("_error"):
            print("FAILED:", mid, "status=", md.get("status"), "body_snip=", md.get("body"))
            continue

        rows = extract_lineup_rows(md)
        df = pd.DataFrame(rows)
        df.to_pickle(out)

        print("Saved:", out, "rows:", len(df))

        await asyncio.sleep(sleep_s)

In [459]:
match_ids

[5161472, 5161473, 5161474, 5161475, 5161476, 5161477, 5161478, 5161479]

In [460]:
match_ids = list(matches_df.match_id)   # sample
await scrape_lineups_fast(page, match_ids, overwrite=True)

Saved: data/lineups/5161472.pkl rows: 40
Saved: data/lineups/5161473.pkl rows: 39
Saved: data/lineups/5161474.pkl rows: 40
Saved: data/lineups/5161475.pkl rows: 40
Saved: data/lineups/5161476.pkl rows: 38
Saved: data/lineups/5161477.pkl rows: 40
Saved: data/lineups/5161478.pkl rows: 40
Saved: data/lineups/5161479.pkl rows: 40


In [461]:
import os
import psycopg2
import pandas as pd
from pathlib import Path

STAGING_PATH = Path("dbt_fotmob/seeds/player_cards_2026_staging.csv")
SEED_PATH    = Path("dbt_fotmob/seeds/player_cards_2026.csv")

staging = pd.read_csv(STAGING_PATH)
active_match_ids = set(staging["match_id"].astype(int))

# Keep only rows where a card was given
new_cards = staging[(staging["yellow_cards"] > 0) | (staging["red_cards"] > 0)].copy()
new_cards = new_cards[["player_id", "match_id", "yellow_cards", "red_cards"]]
new_cards = new_cards.astype({"player_id": int, "match_id": int,
                               "yellow_cards": int, "red_cards": int})

# ---------------------------------------------------------------------------
# Pull existing card history from the DB (source of truth).
# This protects against the CSV being accidentally cleared.
# ---------------------------------------------------------------------------
envrc = Path("../.envrc")
for line in envrc.read_text().splitlines():
    line = line.strip()
    if line.startswith("export "): line = line[len("export "):]
    if "=" in line and not line.startswith("#"):
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip())

conn = psycopg2.connect(
    host=os.environ["NEON_HOST"], user=os.environ["NEON_USER"],
    password=os.environ["NEON_PASSWORD"], dbname=os.environ["NEON_DBNAME"],
    port=int(os.environ.get("NEON_PORT", 5432)), sslmode="require",
)
existing = pd.read_sql(
    "SELECT player_id, match_id, yellow_cards, red_cards FROM fotmob.player_cards_2026",
    conn
)
conn.close()

# Remove rows for current active matches (they'll be replaced by new_cards)
existing = existing[~existing["match_id"].isin(active_match_ids)]

combined = pd.concat([existing, new_cards], ignore_index=True).sort_values(["match_id", "player_id"])
combined.to_csv(SEED_PATH, index=False)
print(f"Written {len(combined)} rows to {SEED_PATH} ({len(new_cards)} new/updated, {len(existing)} preserved from DB)")

Written 76 rows to dbt_fotmob/seeds/player_cards_2026.csv (23 new/updated, 53 preserved from DB)


/var/folders/2z/4k5_nkh1129cf55vhp78fq9m0000gp/T/ipykernel_3130/1458044514.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  existing = pd.read_sql(


In [445]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------------
# Pre-fill player_cards_2026_staging.csv from local lineup pkl files.
# No DB connection needed — reads data/lineups/ and data/matches/2026.csv.
# ---------------------------------------------------------------------------

STAGING_PATH = Path("dbt_fotmob/seeds/player_cards_2026_staging.csv")

active_ids = set(int(l) for l in Path("data/active_match_ids.txt").read_text().splitlines() if l.strip())

# Load match dates from the matches CSV
matches = pd.read_csv("data/matches/2026.csv", usecols=["match_id", "utc_time"])
matches["match_date"] = pd.to_datetime(matches["utc_time"]).dt.date
matches = matches[matches["match_id"].isin(active_ids)]

# Load lineup pkl files for active matches
dfs = []
for mid in active_ids:
    pkl = Path(f"data/lineups/{mid}.pkl")
    if pkl.exists():
        dfs.append(pd.read_pickle(pkl))
    else:
        print(f"  Missing lineup file: {mid}.pkl")

if not dfs:
    print("No lineup files found for active match IDs.")
else:
    lineups = pd.concat(dfs, ignore_index=True)
    lineups = lineups.merge(matches[["match_id", "match_date"]], on="match_id", how="left")
    lineups = lineups.sort_values(["match_date", "team_name", "bucket", "player_name"])
    lineups["yellow_cards"] = 0
    lineups["red_cards"]    = 0

    STAGING_PATH.parent.mkdir(parents=True, exist_ok=True)
    lineups.to_csv(STAGING_PATH, index=False)
    print(f"Pre-filled {len(lineups)} players across {lineups['match_id'].nunique()} matches → {STAGING_PATH}")

Pre-filled 317 players across 8 matches → dbt_fotmob/seeds/player_cards_2026_staging.csv


/var/folders/2z/4k5_nkh1129cf55vhp78fq9m0000gp/T/ipykernel_3130/2635382809.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  lineups = pd.concat(dfs, ignore_index=True)
